# TP 1 — Partie 3 : couche Gold, analyse SQL et ingestion incrémentale

**Objectif :** produire les tables qui répondent aux questions de la direction, puis faire vivre le pipeline quand de nouvelles données arrivent.

La couche Gold contient des tables **orientées usage** : une ligne par commande prête à analyser, et des agrégats directement exploitables par un tableau de bord.

In [ ]:
from pyspark.sql import SparkSession, functions as F, Window

spark = SparkSession.builder.appName("tp1-gold").getOrCreate()
s = lambda table: spark.table(f"lakehouse.silver.{table}")

## Étape 1 — La table centrale `gold.commandes`

Une ligne par commande, avec tout ce qu'il faut pour l'analyser. Complétez :
1. `articles` : par commande, `nb_articles` (nombre de lignes), `montant_produits` (somme de `price`), `montant_fret` (somme de `freight_value`), `nb_vendeurs` (nombre de `seller_id` distincts) ;
2. `paiements` : par commande, `montant_paye` (somme), `mode_paiement` (le `payment_type` du plus gros paiement : `F.max_by("payment_type", "payment_value")`), `nb_echeances` (maximum de `payment_installments`) ;
3. `commandes` : `silver.orders` joint (à gauche) aux deux agrégats et aux colonnes `customer_unique_id`, `customer_city`, `customer_state` des clients, plus une colonne `mois` au format `yyyy-MM`.

In [ ]:
articles = ...   # À COMPLÉTER

paiements = ...  # À COMPLÉTER

commandes = ...  # À COMPLÉTER

commandes.writeTo("lakehouse.gold.commandes").using("iceberg").createOrReplace()
spark.table("lakehouse.gold.commandes").show(3, vertical=True)

## Étape 2 — Les tables d'indicateurs

Écrivez trois requêtes `CREATE OR REPLACE TABLE ... USING iceberg AS SELECT ...`.
**Règle commune :** on exclut les commandes `canceled` et `unavailable`, ainsi que les commandes sans article.

| Table | Une ligne par | Colonnes |
|---|---|---|
| `gold.ventes_mensuelles` | mois | `nb_commandes`, `chiffre_affaires` (somme de `montant_produits`), `panier_moyen`, `nb_clients` (clients uniques) |
| `gold.ventes_categorie` | catégorie (`product_category_en`) | `nb_commandes`, `nb_articles`, `chiffre_affaires`, `prix_moyen` |
| `gold.livraison_etat` | état du client | `nb_livrees`, `delai_moyen_jours`, `delai_estime_moyen_jours`, `taux_retard` (en %) — commandes livrées uniquement |

In [ ]:
FILTRE = "order_status NOT IN ('canceled', 'unavailable') AND nb_articles IS NOT NULL"

spark.sql(f"""
    CREATE OR REPLACE TABLE lakehouse.gold.ventes_mensuelles USING iceberg AS
    -- À COMPLÉTER
""")

spark.sql(f"""
    CREATE OR REPLACE TABLE lakehouse.gold.ventes_categorie USING iceberg AS
    -- À COMPLÉTER (indice : partez de silver.order_items, joignez silver.products et gold.commandes)
""")

spark.sql(f"""
    CREATE OR REPLACE TABLE lakehouse.gold.livraison_etat USING iceberg AS
    -- À COMPLÉTER
""")

spark.table("lakehouse.gold.ventes_mensuelles").show(30)

> **❓ Question 1.** Observez les premiers et les derniers mois de `ventes_mensuelles`. Peut-on les comparer aux autres ? Que proposez-vous pour le tableau de bord ?

## Étape 3 — Répondre à la direction en SQL

Écrivez une requête Spark SQL pour chaque question. Utilisez les tables Gold, et Silver si besoin.

In [ ]:
# Q1. Les 10 catégories qui ont le plus gros chiffre d'affaires
spark.sql("""
    -- À COMPLÉTER
""").show(truncate=False)

In [ ]:
# Q2. La part de chaque mode de paiement (en % des commandes)
spark.sql("""
    -- À COMPLÉTER
""").show()

In [ ]:
# Q3. Le pourcentage de clients qui ont passé plus d'une commande
spark.sql("""
    -- À COMPLÉTER (indice : une ligne par customer_unique_id, puis une moyenne)
""").show()

In [ ]:
# Q4. Les 5 états où les clients attendent le plus longtemps, avec le taux de retard
spark.sql("""
    -- À COMPLÉTER
""").show()

> **❓ Question 2.** Seulement 3 % des clients environ reviennent acheter. Est-ce un problème de qualité des données ou une réalité métier ? Quelle colonne aurait donné un résultat faux, et pourquoi ?

> **❓ Question 3.** Les délais estimés sont presque toujours supérieurs aux délais réels, et pourtant certains états ont un taux de retard élevé. Comment l'expliquez-vous ? Quelle recommandation feriez-vous ?

## Étape 4 — L'ingestion incrémentale

Deux mois et demi ont passé. Le système de vente contient de **nouvelles commandes**, et des commandes « en cours » ont été livrées. Chargez le lot 2 dans PostgreSQL :

In [ ]:
!python /opt/lakehouse/scripts/load_postgres.py --lot 2

On ne relit **pas** toute la table : seulement les lignes modifiées depuis la dernière ingestion, grâce à la colonne `updated_at`. C'est le principe du **filigrane** (*watermark*).

Complétez la cellule :
1. récupérez le plus grand `updated_at` déjà présent dans `bronze.orders` ;
2. lisez en JDBC uniquement les commandes dont `updated_at` est plus récent (option `query` au lieu de `dbtable`) ;
3. **ajoutez** ces lignes à `bronze.orders` (`writeTo(...).append()`) : en Bronze, on n'écrase jamais l'historique.

In [ ]:
import os

JDBC = {
    "url": f"jdbc:postgresql://{os.environ['PG_HOST']}:5432/{os.environ['PG_DB']}",
    "user": os.environ["PG_USER"],
    "password": os.environ["PG_PASSWORD"],
    "driver": "org.postgresql.Driver",
}

filigrane = ...   # À COMPLÉTER
print("dernière ingestion :", filigrane)

nouvelles = ...   # À COMPLÉTER : lecture JDBC des lignes plus récentes que le filigrane

print("lignes nouvelles ou modifiées :", nouvelles.count())
# À COMPLÉTER : ajout à lakehouse.bronze.orders

Il faut maintenant répercuter ces changements en Silver **sans tout recalculer**. L'instruction `MERGE INTO` met à jour les commandes qui existent déjà et insère les nouvelles, en une seule transaction.

La fonction `preparer_orders` reprend les règles de l'étape 5 de la partie 2 (c'est la version de référence). Complétez la requête `MERGE`.

In [ ]:
def preparer_orders(df):
    return (df.drop("_source", "_ingere_le")
        .withColumn("delai_livraison_jours", F.datediff("order_delivered_customer_date", "order_purchase_timestamp"))
        .withColumn("delai_estime_jours", F.datediff("order_estimated_delivery_date", "order_purchase_timestamp"))
        .withColumn("en_retard", F.when(F.col("order_delivered_customer_date").isNotNull(),
                    F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date")))
        .withColumn("dq_livree_sans_date", (F.col("order_status") == "delivered") & F.col("order_delivered_customer_date").isNull())
        .withColumn("dq_transporteur_avant_approbation",
                    F.coalesce(F.col("order_delivered_carrier_date") < F.col("order_approved_at"), F.lit(False))))


avant = spark.sql("SELECT snapshot_id FROM lakehouse.silver.orders.snapshots ORDER BY committed_at DESC LIMIT 1").first()[0]

maj = preparer_orders(nouvelles)
maj.filter("dq_livree_sans_date").writeTo("lakehouse.silver.quarantaine_orders").append()
maj.filter("NOT dq_livree_sans_date").drop("dq_livree_sans_date").createOrReplaceTempView("maj_orders")

spark.sql("""
    -- À COMPLÉTER : MERGE INTO lakehouse.silver.orders ... USING maj_orders ...
""")
print("commandes en Silver :", spark.table("lakehouse.silver.orders").count())

Faites de même pour les articles et les paiements des commandes du lot 2 : on les ajoute d'abord en Bronze, puis on n'**insère** en Silver que les lignes absentes (`WHEN NOT MATCHED THEN INSERT *`), car un article ne change pas une fois créé. La cellule traite les articles ; complétez-la pour les paiements.

In [ ]:
ids = nouvelles.select("order_id").distinct()

def ingerer_nouveaux(table):
    """Lit dans PostgreSQL les lignes des nouvelles commandes, les ajoute en Bronze et les renvoie."""
    df = (spark.read.format("jdbc").options(**JDBC).option("dbtable", f"ventes.{table}").load()
          .join(ids, "order_id", "left_semi")
          .withColumn("_source", F.lit(f"postgres:ventes.{table}"))
          .withColumn("_ingere_le", F.current_timestamp()))
    df.writeTo(f"lakehouse.bronze.{table}").append()
    return df.drop("_source", "_ingere_le")

(ingerer_nouveaux("order_items")
    .withColumn("price", F.col("price").cast("double"))
    .withColumn("freight_value", F.col("freight_value").cast("double"))
    .createOrReplaceTempView("maj_items"))
spark.sql("""
    MERGE INTO lakehouse.silver.order_items AS cible
    USING maj_items AS source
    ON cible.order_id = source.order_id AND cible.order_item_id = source.order_item_id
    WHEN NOT MATCHED THEN INSERT *
""")

# À COMPLÉTER : même chose pour order_payments (sans les paiements 'not_defined')

print("articles :", spark.table("lakehouse.silver.order_items").count())
print("paiements :", spark.table("lakehouse.silver.order_payments").count())

## Étape 5 — Voyager dans le temps

Iceberg a gardé la version de `silver.orders` d'**avant** le `MERGE` (variable `avant`). Comparez la répartition des statuts entre les deux versions.

In [ ]:
spark.sql("""
    SELECT committed_at, snapshot_id, operation, summary['added-records'] AS ajoutees
    FROM lakehouse.silver.orders.snapshots
    ORDER BY committed_at
""").show(truncate=False)

spark.sql(f"""
    SELECT a.order_status, a.nb AS avant_merge, b.nb AS apres_merge
    FROM (SELECT order_status, COUNT(*) AS nb FROM lakehouse.silver.orders VERSION AS OF {avant} GROUP BY order_status) a
    FULL JOIN (SELECT order_status, COUNT(*) AS nb FROM lakehouse.silver.orders GROUP BY order_status) b
      ON a.order_status = b.order_status
    ORDER BY apres_merge DESC
""").show()

> **❓ Question 4.** Le nombre de commandes `shipped` a fortement baissé après le `MERGE`. Pourquoi ? En quoi le *time travel* est-il utile en entreprise ?

## Étape 6 — Mettre à jour Gold

Réexécutez les cellules des **étapes 1 et 2** : les tables Gold sont recalculées entièrement à partir de Silver, qui est maintenant à jour.

> **❓ Question 5.** Pourquoi recalcule-t-on Gold entièrement alors qu'on a fait un `MERGE` incrémental en Silver ? Dans quel cas faudrait-il aussi rendre Gold incrémental ?

## Étape 7 — Le tableau de bord

Passez au fichier `apps/dashboard_tp1.py` et suivez les consignes du support (partie 4).